<a href="https://colab.research.google.com/github/adcgunwoo/Titantic/blob/main/titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
# 행렬 연산 관련
import pandas as pd
import numpy as np

# 데이터 시각화 관련
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set(style = 'white', context = 'notebook', palette = 'deep')

# 경고 무시
import warnings
warnings.filterwarnings('ignore')

# 파일 입출력 관련. 제출 파일 만들때 씀
import glob
import os
# 정규식
import re

import statsmodels.api as sm
from sklearn import preprocessing

# 이미 구현된 인공지능 분류 모델, 학습 기법들
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC


from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold, learning_curve

In [36]:
from google import colab
from google.colab import drive
drive.mount('/content/gdrive')
colab_path="/content/gdrive/MyDrive/KISIA/Data/"


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [37]:
df_train = pd.read_csv(colab_path+'train.csv') #df는 data frame의 약자
df_test = pd.read_csv(colab_path+'test.csv') #pd는 pandas의 약
test_y=pd.read_csv(colab_path+'submission.csv')
test_y=test_y['Survived']
print("학습데이터: {}건, 테스트용 데이터: {}건".format(len(df_train), len(df_test)))


학습데이터: 891건, 테스트용 데이터: 418건


In [38]:
df_train

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [39]:
df_train.head() #head메서드 처음 5번째 인원

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [40]:
df_train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [41]:
df_test.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [42]:
df_train.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [43]:
df_train.info()  #Non-Null Count의 숫자의 의미, Age, Cabin, Embarked는 왜 Non-Null Count가 다를까?

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [44]:
df_train.describe() #object 컬럼은 빠졌다.
#df_train['Age'].describe -> age만 보고싶다.

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [45]:
df_train.shape

(891, 12)

In [46]:
#data셋 전처리 한번에 하기
dataset = [df_train, df_test] #데이터셋 병합
#for data in dataset:
# data.rename(columns = {'Sex' : 'Gender'}, inplace = True)
dataset

[     PassengerId  Survived  Pclass  \
 0              1         0       3   
 1              2         1       1   
 2              3         1       3   
 3              4         1       1   
 4              5         0       3   
 ..           ...       ...     ...   
 886          887         0       2   
 887          888         1       1   
 888          889         0       3   
 889          890         1       1   
 890          891         0       3   
 
                                                   Name     Sex   Age  SibSp  \
 0                              Braund, Mr. Owen Harris    male  22.0      1   
 1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
 2                               Heikkinen, Miss. Laina  female  26.0      0   
 3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
 4                             Allen, Mr. William Henry    male  35.0      0   
 ..                                               

In [47]:
sum(df_train['Survived']==0)/len(df_train['Survived']) #사망자 비율

0.6161616161616161

In [48]:
df_train['Embarked'].value_counts()

,count
Embarked,
S,644
C,168
Q,77


In [49]:
df_train[df_train['Embarked'].isnull()] #탑승지가 비어있음

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN


In [73]:
df_train.groupby(['Pclass','Embarked'])['Fare'].mean() #등급별 그리고 탑승지별 요금

Pclass  Embarked
1       C           104.718529
        Q            90.000000
        S            70.364862
2       C            25.358335
        Q            12.350000
        S            20.327439
3       C            11.214083
        Q            11.183393
        S            14.644083
Name: Fare, dtype: float64

In [50]:
df_train.groupby('Embarked')['Pclass'].value_counts() #groupby는 어떤 함수일까? column을 합쳐서 볼 수 있나보다.

Embarked  Pclass
C         1          85
          3          66
          2          17
Q         3          72
          2           3
          1           2
S         3         353
          2         164
          1         127
Name: count, dtype: int64

In [77]:
df_train['Fare'].describe() #요금 평균

,Fare
count,891.000000
mean,32.204208
std,49.693429
min,0.000000
25%,7.910400
50%,14.454200
75%,31.000000
max,512.329200


In [79]:
df_train[df_train['Fare'] ==  512.329200] #가장 비싼 돈을 주고 탄 사람들

,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_isna,Title
258,259,1.0,1,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,NaN,C,0.0,Miss
679,680,1.0,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,512.3292,B51 B53 B55,C,0.0,Mr
737,738,1.0,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,512.3292,B101,C,0.0,Mr
343,1235,NaN,1,"Cardeza, Mrs. James Warburton Martinez (Charlo...",female,58.0,0,1,PC 17755,512.3292,B51 B53 B55,C,NaN,NaN


In [80]:
df_alldata[df_alldata['Fare'] ==  512.329200] #데이터를 합친 전부(test와 train을 합침), 마지막 여성은 우리가 예측해야

,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_isna,Title
258,259,1.0,1,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,NaN,C,0.0,Miss
679,680,1.0,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,512.3292,B51 B53 B55,C,0.0,Mr
737,738,1.0,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,512.3292,B101,C,0.0,Mr
343,1235,NaN,1,"Cardeza, Mrs. James Warburton Martinez (Charlo...",female,58.0,0,1,PC 17755,512.3292,B51 B53 B55,C,NaN,NaN


In [81]:
df_train['Ticket'].value_counts() #티켓 번호 조회, 같은 티켓 번호가 동일한것들이 있다.

,count
Ticket,
347082,7
1601,7
CA. 2343,7
3101295,6
CA 2144,6
...,...
PC 17590,1
17463,1
330877,1


In [82]:
df_alldata['Ticket'].value_counts()

,count
Ticket,
CA. 2343,11
1601,8
CA 2144,8
347082,7
S.O.C. 14879,7
...,...
A.5. 3236,1
347086,1
365237,1


In [84]:
df_alldata[df_alldata['Ticket'] == 'CA. 2343'] #같은 표를 산 인원들(train+test) 성이 다들 같네, Sibsp+Parch는 10으로 동일하다.
#여기서 나까지 포함해서 총 11명인 것이다.

,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_isna,Title
159,160,0.0,3,"Sage, Master. Thomas Henry",male,4.574167,8,2,CA. 2343,69.55,NaN,S,1.0,Master
180,181,0.0,3,"Sage, Miss. Constance Gladys",female,21.773973,8,2,CA. 2343,69.55,NaN,S,1.0,Miss
201,202,0.0,3,"Sage, Mr. Frederick",male,32.368090,8,2,CA. 2343,69.55,NaN,S,1.0,Mr
324,325,0.0,3,"Sage, Mr. George John Jr",male,32.368090,8,2,CA. 2343,69.55,NaN,S,1.0,Mr
792,793,0.0,3,"Sage, Miss. Stella Anna",female,21.773973,8,2,CA. 2343,69.55,NaN,S,1.0,Miss
846,847,0.0,3,"Sage, Mr. Douglas Bullen",male,32.368090,8,2,CA. 2343,69.55,NaN,S,1.0,Mr
863,864,0.0,3,"Sage, Miss. Dorothy Edith ""Dolly""",female,21.773973,8,2,CA. 2343,69.55,NaN,S,1.0,Miss
188,1080,NaN,3,"Sage, Miss. Ada",female,NaN,8,2,CA. 2343,69.55,NaN,S,NaN,NaN
342,1234,NaN,3,"Sage, Mr. John George",male,NaN,1,9,CA. 2343,69.55,NaN,S,NaN,NaN
360,1252,NaN,3,"Sage, Master. William Henry",male,14.500000,8,2,CA. 2343,69.55,NaN,S,NaN,NaN


In [51]:
df=pd.concat(dataset, axis=0) #무슨 메서드일까
print(len(df))
df.reset_index(drop=True)

1309


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,NaN,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
1305,1306,NaN,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
1306,1307,NaN,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
1307,1308,NaN,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [52]:
replace_value=df_train['Embarked'].value_counts(dropna=False).idxmax() #무슨 메서드일까
replace_value

'S'

In [88]:
# 결측치 처리(대체) - Embarked
df_train['Embarked'] = df_train['Embarked'].fillna(replace_value) #null 자리에 'S'를 채워넣겠다는 의미

In [89]:
df_train['Embarked'].isna().sum() #nan인게 없어진것을 확인할 수 있다.

np.int64(0)

In [90]:
df_train[df_train['Embarked'].isna()]['Survived']==1 #뭘 나타낼까?

,Survived


In [53]:
sum(df_train['Age'].isna()) #isna() 메서드는 뭘까

177

In [96]:
df_train[df_train['Age'].isna()]['Survived'].mean()

nan

In [97]:
df_train['Age'].isna().astype(int)

,Age
0,0
1,0
2,0
3,0
4,0
...,...
886,0
887,0
888,0
889,0


In [98]:
df_train['Age_isna'] = df_train['Age'].isna().astype(int)
df_train

,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_isna,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000000,1,0,A/5 21171,7.2500,NaN,S,0,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000000,1,0,PC 17599,71.2833,C85,C,0,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000000,0,0,STON/O2. 3101282,7.9250,NaN,S,0,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000000,1,0,113803,53.1000,C123,S,0,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.000000,0,0,373450,8.0500,NaN,S,0,Mr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.000000,0,0,211536,13.0000,NaN,S,0,Rev
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.000000,0,0,112053,30.0000,B42,S,0,Miss
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,21.773973,1,2,W./C. 6607,23.4500,NaN,S,0,Miss
889,890,1,1,"Behr, Mr. Karl Howell",male,26.000000,0,0,111369,30.0000,C148,C,0,Mr


In [54]:
df_train[df_train['Age'].isna()]['Survived']==1 #뭘 나타낼까?
#sum(df_train[df_train['Age'].isna()]['Survived']==1)/177


,Survived
5,False
17,True
19,True
26,False
28,True
...,...
859,False
863,False
868,False
878,False


In [55]:
df_train['Age_isna']=df_train['Age'].isna().astype(int)
df_train
#나이가 Nan인 사람들인데 이름 가운데에 master(존칭)는 어리겠지 ex. 도련님

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_isna
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,0
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,0
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,1
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,0


In [56]:
df_train[df_train['Name'].str.contains('Master')].describe() #master 호칭(어린 남성)은 가진 사람의 나이와 동승자 숫자 그리고 생존률을 유심히 봐라

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Age_isna
count,40.000000,40.000000,40.000000,36.000000,40.000000,40.000000,40.000000,40.000000
mean,414.975000,0.575000,2.625000,4.574167,2.300000,1.375000,34.703125,0.100000
std,301.717518,0.500641,0.627878,3.619872,1.910833,0.540062,28.051752,0.303822
min,8.000000,0.000000,1.000000,0.420000,0.000000,0.000000,8.516700,0.000000
25%,165.750000,0.000000,2.000000,1.000000,1.000000,1.000000,18.750000,0.000000
50%,345.000000,1.000000,3.000000,3.500000,1.000000,1.000000,29.062500,0.000000
75%,764.000000,1.000000,3.000000,8.000000,4.000000,2.000000,39.171875,0.000000
max,870.000000,1.000000,3.000000,12.000000,8.000000,2.000000,151.550000,1.000000


In [57]:
df_train[df_train['Name'].str.contains('Mr\.')].describe()
#이름 가운데에 Mr 있는 애들은 생존률이 15%밖에 안된다.

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Age_isna
count,517.000000,517.000000,517.000000,398.000000,517.000000,517.000000,517.000000,517.000000
mean,454.499033,0.156673,2.410058,32.368090,0.288201,0.152805,24.441560,0.230174
std,253.715526,0.363844,0.810622,12.708793,0.821298,0.533615,44.378561,0.421352
min,1.000000,0.000000,1.000000,11.000000,0.000000,0.000000,0.000000,0.000000
25%,226.000000,0.000000,2.000000,23.000000,0.000000,0.000000,7.800000,0.000000
50%,466.000000,0.000000,3.000000,30.000000,0.000000,0.000000,9.350000,0.000000
75%,674.000000,0.000000,3.000000,39.000000,0.000000,0.000000,26.000000,0.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,5.000000,512.329200,1.000000


In [58]:
df_train[df_train['Name'].str.contains('Mrs\.')].describe()
#이름 가운데에 Mrs인 애들은 생존률이 80%이다.

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Age_isna
count,125.000000,125.00000,125.000000,108.000000,125.000000,125.000000,125.000000,125.000000
mean,453.160000,0.79200,2.000000,35.898148,0.696000,0.832000,45.138533,0.136000
std,270.762764,0.40751,0.823055,11.433628,0.598708,1.274666,45.723716,0.344168
min,2.000000,0.00000,1.000000,14.000000,0.000000,0.000000,7.225000,0.000000
25%,255.000000,1.00000,1.000000,27.750000,0.000000,0.000000,15.850000,0.000000
50%,438.000000,1.00000,2.000000,35.000000,1.000000,0.000000,26.000000,0.000000
75%,679.000000,1.00000,3.000000,44.000000,1.000000,1.000000,57.000000,0.000000
max,886.000000,1.00000,3.000000,63.000000,3.000000,6.000000,247.520800,1.000000


In [59]:
df_train_TitleAge = df_train
df_train_TitleAge['Title'] = df_train['Name'].apply(lambda x:x.split(',')[1].split('.')[0].strip())
#print(df_train['Title'].isnull().value_counts())
title_age_mean = df_train_TitleAge.groupby('Title')['Age'].mean().to_dict()
title_age_replace_value = df_train_TitleAge['Title'].map(title_age_mean)
df_train_TitleAge['Age'] = df_train_TitleAge['Age'].fillna(title_age_replace_value)      #얘는 진짜 모르겠다

In [60]:
df_train_TitleAge = df_train
df_train_TitleAge['Title'] = df_train['Name'].apply(lambda x:x.split(',')[1].split('.')[0].strip())
df_train_TitleAge['Title'].value_counts() #가운데 이름을 나타낸건가?

,count
Title,
Mr,517
Miss,182
Mrs,125
Master,40
Dr,7
Rev,6
Col,2
Mlle,2
Major,2


In [61]:
df_train[df_train['Name'].str.contains('Miss\.')].describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Age_isna
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000
mean,408.884615,0.697802,2.307692,21.773973,0.714286,0.549451,43.797873,0.197802
std,246.775812,0.460477,0.849989,11.626892,1.431961,0.804184,66.027199,0.399441
min,3.000000,0.000000,1.000000,0.750000,0.000000,0.000000,6.750000,0.000000
25%,213.000000,0.000000,1.250000,16.000000,0.000000,0.000000,7.951050,0.000000
50%,381.500000,1.000000,3.000000,21.773973,0.000000,0.000000,15.620850,0.000000
75%,612.250000,1.000000,3.000000,26.750000,1.000000,1.000000,41.034400,0.000000
max,889.000000,1.000000,3.000000,63.000000,8.000000,2.000000,512.329200,1.000000


In [62]:
replace_value = df_train['Age'].mean().round(2)
replace_value #나이 그냥 있는 것들의 평균으로 계산해서 결측치를 처리한다.(머리 쓰기 포기하는 방법)

np.float64(29.75)

In [63]:
# 결측치 처리(대체) - Age
df_train['Age'] = df_train['Age'].fillna(replace_value)

In [64]:
#결측치 처리 - Cabin
df_train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'Age_isna', 'Title'],
      dtype='object')

In [65]:
df_merged_train = pd.concat([df_train[df_train['Name'].str.contains('Master')].fillna(5), df_train[df_train['Name'].str.contains('Master')==False].fillna(29)], axis=0) #왜 안되니
df_merged_train.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [66]:
dataset = [df_train, df_test]
for data in dataset:
    data.rename(columns = {'Sex' : 'Gender'}, inplace=True)
dataset
df_alldata = pd.concat(dataset, axis=0)
df_alldata

,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_isna,Title
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0.0,Mr
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,0.0,Mrs
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0.0,Miss
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,0.0,Mrs
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0.0,Mr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,NaN,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S,NaN,NaN
414,1306,NaN,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,NaN,NaN
415,1307,NaN,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,NaN,NaN
416,1308,NaN,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S,NaN,NaN


In [68]:
print((df_train['Survived'].value_counts()/len(df_train)*100).round(2)) #훈련데이터의 생존자 사망자 비율

Survived
0    61.62
1    38.38
Name: count, dtype: float64


In [70]:
#print((df_test['Survived'].value_counts()/len(df_train)*100).round(2)) 아직 NUll이어서 불가

In [72]:
#결측치 확인
print('### train dataset ###')
print(df_train.isnull().mean()) #결측치의 개수, age가 왜 안뜨지
print('### test data set ###')
print(df_test.isnull().mean()) #결측치의 개수

### train dataset ###
PassengerId    0.000000
Survived       0.000000
Pclass         0.000000
Name           0.000000
Gender         0.000000
Age            0.000000
SibSp          0.000000
Parch          0.000000
Ticket         0.000000
Fare           0.000000
Cabin          0.771044
Embarked       0.002245
Age_isna       0.000000
Title          0.000000
dtype: float64
### test data set ###
PassengerId    0.000000
Pclass         0.000000
Name           0.000000
Gender         0.000000
Age            0.205742
SibSp          0.000000
Parch          0.000000
Ticket         0.000000
Fare           0.002392
Cabin          0.782297
Embarked       0.000000
dtype: float64
